# Indian Retail Sales and Revenue Analytics
## Exploratory Data Analysis (DAV Lab AAT)

**Student Name:** [Your Name]  
**USN:** [Your USN]  
**Subject:** Data Analytics and Visualization (DAV) Lab  

---
### Notebook Overview:
In this notebook, we perform simple step-by-step exploratory data analysis on the Indian Retail dataset:
1. Load the raw dataset using `pandas`.
2. Inspect data types, missing values, and duplicate rows.
3. Clean the data using standard Pandas functions (`fillna`, `drop_duplicates`, filtering).
4. Calculate key metrics (Total Revenue, Total Orders, Average Order Value).
5. Group data by Category, State, and Month using `groupby()`.
6. Visualize sales trends using simple Matplotlib and Seaborn charts.

In [ ]:
# Step 1: Import standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set chart size
plt.rcParams['figure.figsize'] = (10, 5)

print("Libraries imported successfully!")

## Step 2: Load the Dataset
We load the raw CSV file using `pd.read_csv()`.

In [ ]:
# Load the dataset
# (Note: Using Latin-1 fallback ensures no encoding errors)
try:
    df = pd.read_csv('../data/raw/indian_retail_orders.csv')
except UnicodeDecodeError:
    df = pd.read_csv('../data/raw/indian_retail_orders.csv', encoding='latin1')

print("Dataset Shape:", df.shape)
df.head()

## Step 3: Inspect Data Types and Summary Statistics
We check column types using `df.info()` and statistical summaries using `df.describe()`.

In [ ]:
# Check column types and non-null counts
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe().round(2)

## Step 4: Check for Data Quality Issues
Before analyzing, we check:
1. Missing values (`isnull().sum()`)
2. Duplicate records (`duplicated().sum()`)
3. Negative quantities (data entry errors)

In [ ]:
# 1. Check missing values per column
print("--- Missing Values ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

# 2. Check duplicate rows
print("\nTotal Duplicate Rows:", df.duplicated().sum())

# 3. Check for negative quantities
neg_qty = (df['quantity'] <= 0).sum()
print("Rows with Quantity <= 0:", neg_qty)

## Step 5: Data Cleaning
We clean the data step-by-step:
- Drop duplicate rows using `drop_duplicates()`
- Convert `order_date` to datetime using `pd.to_datetime()`
- Remove bad date years (outside 2020–2026) and non-positive quantities
- Fill missing discounts with 0 and missing text fields with 'Unknown'
- Add a `year_month` column for monthly analysis

In [ ]:
# 1. Remove duplicate rows
df = df.drop_duplicates()

# 2. Convert order_date to datetime
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

# 3. Filter valid dates and positive quantities
df = df[(df['order_date'].dt.year >= 2020) & (df['order_date'].dt.year <= 2026)]
df = df[df['quantity'] > 0]

# 4. Fill missing values
df['discount_percent'] = df['discount_percent'].fillna(0)
df['payment_method'] = df['payment_method'].fillna('Unknown')
df['city'] = df['city'].fillna('Unknown')
df['customer_name'] = df['customer_name'].fillna('Unknown')

# 5. Add year_month column for monthly trend analysis
df['year_month'] = df['order_date'].dt.to_period('M').astype(str)
df['revenue'] = df['sales_amount']

print("Cleaned Data Shape:", df.shape)
print("Data cleaning completed successfully!")

## Step 6: Calculate Key Business Metrics
We calculate the foundational retail KPIs:
- **Total Revenue:** Sum of all sales
- **Total Orders:** Count of distinct `order_id` values (using `nunique()` to avoid double-counting)
- **Total Units Sold:** Sum of `quantity`
- **Average Order Value (AOV):** Total Revenue / Total Distinct Orders

In [ ]:
# Calculate KPIs
total_revenue = df['revenue'].sum()
total_orders = df['order_id'].nunique()
total_units = df['quantity'].sum()
aov = total_revenue / total_orders
total_profit = df['profit'].sum()
profit_margin = (total_profit / total_revenue) * 100

print(f"Total Gross Revenue : Rs. {total_revenue:,.2f}  (Rs. {total_revenue/1e7:.2f} Cr)")
print(f"Total Distinct Orders: {total_orders:,}")
print(f"Total Units Sold    : {total_units:,}")
print(f"Average Order Value : Rs. {aov:,.2f}")
print(f"Total Profit        : Rs. {total_profit:,.2f}  (Margin: {profit_margin:.2f}%)")

## Step 7: Category-wise Revenue Analysis
We use `groupby('category')` to find which merchandise category generates the most revenue.

In [ ]:
# Group by Category
cat_revenue = df.groupby('category')['revenue'].sum().sort_values(ascending=False)

# Display as table with Crores
cat_table = pd.DataFrame({
    'Total Revenue (Rs)': cat_revenue,
    'Revenue (Cr)': (cat_revenue / 1e7).round(2),
    'Share (%)': ((cat_revenue / total_revenue) * 100).round(2)
})
cat_table

In [ ]:
# Horizontal Bar Chart for Category Revenue
plt.figure(figsize=(10, 5))
cat_table['Revenue (Cr)'].sort_values().plot(kind='barh', color='teal')
plt.title('Revenue by Product Category (Rs. Crores)', fontsize=14, fontweight='bold')
plt.xlabel('Revenue (Rs. Crores)')
plt.ylabel('Category')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Step 8: Monthly Revenue Trend
We analyze how sales change across months using a simple line plot.

In [ ]:
# Group by Year-Month
monthly_revenue = df.groupby('year_month')['revenue'].sum() / 1e7

# Plot Monthly Trend Line
plt.figure(figsize=(12, 5))
monthly_revenue.plot(kind='line', marker='o', color='royalblue', linewidth=2)
plt.title('Monthly Sales Trend (2022 - 2024)', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Revenue (Rs. Crores)')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## Step 9: Top 10 Indian States by Sales
We find the leading states using `groupby('state')`.

In [ ]:
# Top 10 States by Revenue
top_states = df.groupby('state')['revenue'].sum().sort_values(ascending=False).head(10) / 1e7

plt.figure(figsize=(10, 5))
top_states.sort_values().plot(kind='barh', color='coral')
plt.title('Top 10 Indian States by Revenue (Rs. Crores)', fontsize=14, fontweight='bold')
plt.xlabel('Revenue (Rs. Crores)')
plt.ylabel('State')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Step 10: Payment Method Distribution
We check customer payment preferences using a simple pie chart.

In [ ]:
# Order count by payment method
payment_counts = df['payment_method'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(payment_counts, labels=payment_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
plt.title('Payment Method Breakdown', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 11: Correlation Analysis
We compute correlations between numeric variables using `df.corr()` and display a Seaborn heatmap.

In [ ]:
# Correlation matrix of numeric fields
numeric_fields = ['quantity', 'unit_price', 'discount_percent', 'sales_amount', 'profit']
corr_matrix = df[numeric_fields].corr()

plt.figure(figsize=(8, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 12: Key Findings for Viva
Here is a concise summary of the findings to present to the faculty:
1. **Total Revenue:** Rs. 22.06 Crores across 14,970 distinct orders.
2. **Average Order Value (AOV):** Rs. 14,738 per order.
3. **Leading Category:** **Electronics** is the #1 category, accounting for ~75% of revenue due to high unit prices (laptops, phones).
4. **Leading State:** **Maharashtra** is the top-performing state (~14.5% share), followed by Delhi and Uttar Pradesh.
5. **Preferred Payment Method:** **UPI** is the most popular payment method (~35% of transactions), followed by Cash on Delivery (~24%).
6. **Seasonality:** Peak monthly revenue occurs around year-end festive months (October–December).